In [1]:
# ==========================================
# PHASE 5.1 - ADVANCED CUSTOMER FEATURES
# ==========================================

import pandas as pd
import numpy as np

# ------------------------------------------
# Load Cleaned Dataset
# ------------------------------------------

df = pd.read_csv("../data/cleaned_online_retail.csv")

# Convert date
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("Dataset Shape:", df.shape)

# ------------------------------------------
# Snapshot Date
# ------------------------------------------

snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

# ------------------------------------------
# Customer Features
# ------------------------------------------

customer_features = df.groupby("CustomerID").agg({

    # Recency
    "InvoiceDate": [
        lambda x: (snapshot_date - x.max()).days,
        "min",
        "max"
    ],

    # Frequency
    "InvoiceNo": "nunique",

    # Revenue
    "Revenue": [
        "sum",
        "mean"
    ],

    # Quantity
    "Quantity": [
        "sum",
        "mean"
    ]

})

# ------------------------------------------
# Flatten Columns
# ------------------------------------------

customer_features.columns = [

    "Recency",
    "FirstPurchase",
    "LastPurchase",

    "Frequency",

    "TotalRevenue",
    "AvgOrderValue",

    "TotalQuantity",
    "AvgQuantity"

]

customer_features = customer_features.reset_index()

# ------------------------------------------
# Active Days
# ------------------------------------------

customer_features["ActiveDays"] = (

    customer_features["LastPurchase"] -
    customer_features["FirstPurchase"]

).dt.days

# ------------------------------------------
# Remove Date Columns
# ------------------------------------------

customer_features.drop(
    columns=["FirstPurchase", "LastPurchase"],
    inplace=True
)

# ------------------------------------------
# Log Revenue
# ------------------------------------------

customer_features["LogRevenue"] = np.log1p(
    customer_features["TotalRevenue"]
)

print("\nCustomer Dataset Shape:")
print(customer_features.shape)

print("\nColumns:")
print(customer_features.columns)

print("\nFirst 5 Rows:")
print(customer_features.head())

# ------------------------------------------
# Save
# ------------------------------------------

customer_features.to_csv(
    "../data/advanced_customer_features.csv",
    index=False
)

print("\nAdvanced Features Saved!")

Dataset Shape: (392692, 9)

Customer Dataset Shape:
(4338, 9)

Columns:
Index(['CustomerID', 'Recency', 'Frequency', 'TotalRevenue', 'AvgOrderValue',
       'TotalQuantity', 'AvgQuantity', 'ActiveDays', 'LogRevenue'],
      dtype='object')

First 5 Rows:
   CustomerID  Recency  Frequency  TotalRevenue  AvgOrderValue  TotalQuantity  \
0     12346.0      326          1      77183.60   77183.600000          74215   
1     12347.0        2          7       4310.00      23.681319           2458   
2     12348.0       75          4       1797.24      57.975484           2341   
3     12349.0       19          1       1757.55      24.076027            631   
4     12350.0      310          1        334.40      19.670588            197   

    AvgQuantity  ActiveDays  LogRevenue  
0  74215.000000           0   11.253955  
1     13.505495         365    8.368925  
2     75.516129         282    7.494564  
3      8.643836           0    7.472245  
4     11.588235           0    5.815324  

Advan